We will:

1. Take two face images

1. Convert each face into a vector (embedding)

1. Compare the two vectors

1. Print similarity score

### What We Are Actually Building

Imagine:

`image1.jpg → [128 numbers]` <br>
`image2.jpg → [128 numbers]`


Then:

    compare numbers
    if similar → same person
    if different → different person


That’s face recognition at its core.

### What Is Embedding?

Embeddings are numerical representations of data such as: 
- text
- images
- audio

This data is converted into vectors of floating-point numbers. 

### Step 1 — Install Simple Face Embedding Library

We’ll use:

    facenet


- It’s built on dlib. 
- Very stable. 
- Easy to use. 

Install:

    pip install facenet-pytorch


### Step 2 — Basic Code

In [1]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import numpy as np


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn = MTCNN(image_size=160, margin=0, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

Explanation

- image_size=160 <br>
→ face will be resized to 160×160 before embedding

- pretrained='vggface2' <br>
→ model trained on large real face dataset

- .eval() <br>
→ turn off dropout, batchnorm training behavior

#### What Is InceptionResnetV1?

Simple explanation:

- It is a deep CNN
- It uses residual connections (like ResNet)
- It was trained on millions of face images
- Its goal is NOT classification
- Its goal is embedding learning

Very important:

This model does NOT output:

`Person A`
`Person B`

It outputs:

`512 numbers`

### Step 3 — Load Two Images

Put two face images in the same folder.

Then:

In [3]:
img1 = Image.open("faces/single/amitabh_bachchan/ab1.png")
img2 = Image.open("faces/single/amitabh_bachchan/ab2.png")

### Step 4 — Detect and Crop Face

In [4]:
face1 = mtcnn(img1)
face2 = mtcnn(img2)

What happens here:

- Detect face
- Align it
- Resize to 160×160
- Convert to tensor

Now face1 is already a tensor ready for the model.

### Step 5 — Get Embeddings

In [5]:
face1 = face1.unsqueeze(0).to(device)
face2 = face2.unsqueeze(0).to(device)

with torch.no_grad():
    emb1 = resnet(face1)
    emb2 = resnet(face2)

print(resnet)

InceptionResnetV1(
  (conv2d_1a): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2a): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2b): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (maxpool_3a): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2d_3b): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_4a): 

What is `.unsqueeze(0)`?

Original shape:

> (3, 160, 160)

Model expects:

> (batch_size, channels, height, width)

So we add batch dimension:

> (1, 3, 160, 160)

In [6]:
print(emb1.shape)

torch.Size([1, 512])


### Step 6 — Compute Distance

In [7]:
distance = torch.norm(emb1 - emb2).item()
print("Distance: ", distance)

Distance:  0.3925554156303406


Lower distance → more similar.